
# Day 05: Model Improvement and Feature Selection

## Objectives
- Review previous model evaluation results.
- Explore feature selection and feature reduction.
- Evaluate model performance without the absences feature.
- Compare the updated model with the original Random Forest.
- Finalize the prediction workflow.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import cross_val_score

# Load preprocessed data
data = joblib.load("../artifacts/processed_data.pkl")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

# Load original selected model
original_model = joblib.load(
    "../artifacts/student_performance_model.pkl"
)

# Load preprocessor
preprocessor = joblib.load(
    "../artifacts/preprocessor.pkl"
)

print("Day 05 Environment Loaded Successfully")
print("--------------------------------------")
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Original model:", type(original_model).__name__)
print("Preprocessor loaded:", preprocessor is not None)


Day 05 Environment Loaded Successfully
--------------------------------------
Training data: (316, 56)
Testing data: (79, 56)
Original model: RandomForestRegressor
Preprocessor loaded: True


### 2. Baseline Performance Reference

In [2]:

# Original model performance reference

original_predictions = original_model.predict(X_test)

baseline_results = {
    "Model": "Original Random Forest",
    "Features": X_train.shape[1],
    "MAE": mean_absolute_error(y_test, original_predictions),
    "RMSE": np.sqrt(
        mean_squared_error(y_test, original_predictions)
    ),
    "R2": r2_score(y_test, original_predictions)
}

baseline_df = pd.DataFrame([baseline_results])

display(
    baseline_df.round(3)
)


,Model,Features,MAE,RMSE,R2
0,Original Random Forest,56,3.166,3.934,0.245


### 3. Experiment B – Model Without Absences

In [4]:

# Identify the absences feature position
feature_names = preprocessor.get_feature_names_out()

absences_feature = "num__absences"

absences_index = list(feature_names).index(absences_feature)

print("Absences feature index:", absences_index)

# Remove absences column using NumPy indexing
X_train_no_abs = np.delete(
    X_train,
    absences_index,
    axis=1
)

X_test_no_abs = np.delete(
    X_test,
    absences_index,
    axis=1
)

print("\nFeature Removal Summary")
print("-----------------------")
print("Original training shape:", X_train.shape)
print("Updated training shape:", X_train_no_abs.shape)
print("Original testing shape:", X_test.shape)
print("Updated testing shape:", X_test_no_abs.shape)
print("Absences removed successfully")


Absences feature index: 55

Feature Removal Summary
-----------------------
Original training shape: (316, 56)
Updated training shape: (316, 55)
Original testing shape: (79, 56)
Updated testing shape: (79, 55)
Absences removed successfully


### 4. Train Random Forest Without Absences

In [5]:

# Train Random Forest without absences
rf_no_abs = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42
)

rf_no_abs.fit(X_train_no_abs, y_train)

# Generate predictions
predictions_no_abs = rf_no_abs.predict(X_test_no_abs)

# Evaluate model
no_abs_mae = mean_absolute_error(
    y_test, predictions_no_abs
)

no_abs_rmse = np.sqrt(
    mean_squared_error(y_test, predictions_no_abs)
)

no_abs_r2 = r2_score(
    y_test, predictions_no_abs
)

print("Random Forest Without Absences")
print("--------------------------------")
print("MAE:", round(no_abs_mae, 3))
print("RMSE:", round(no_abs_rmse, 3))
print("R2 Score:", round(no_abs_r2, 3))


Random Forest Without Absences
--------------------------------
MAE: 3.347
RMSE: 4.202
R2 Score: 0.139


### 5. Cross-Validation Without Absences

In [6]:

# 5-fold cross-validation without absences
cv_scores_no_abs = cross_val_score(
    rf_no_abs,
    X_train_no_abs,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

no_abs_cv_mae = -cv_scores_no_abs.mean()
no_abs_cv_std = cv_scores_no_abs.std()

print("Cross-Validation Without Absences")
print("---------------------------------")
print("Fold MAE Scores:", np.round(-cv_scores_no_abs, 3))
print("Mean CV MAE:", round(no_abs_cv_mae, 3))
print("CV Standard Deviation:", round(no_abs_cv_std, 3))


Cross-Validation Without Absences
---------------------------------
Fold MAE Scores: [3.471 2.951 3.181 3.53  2.758]
Mean CV MAE: 3.178
CV Standard Deviation: 0.296


### 6. Original vs Without Absences – Comparison

In [8]:

comparison_results = pd.DataFrame([
    {
        "Model": "Original Random Forest",
        "Features": 56,
        "Test MAE": baseline_results["MAE"],
        "Test RMSE": baseline_results["RMSE"],
        "Test R2": baseline_results["R2"],
        "CV MAE": 2.959,
        "CV Std": 0.198
    },
    {
        "Model": "Random Forest Without Absences",
        "Features": 55,
        "Test MAE": no_abs_mae,
        "Test RMSE": no_abs_rmse,
        "Test R2": no_abs_r2,
        "CV MAE": no_abs_cv_mae,
        "CV Std": no_abs_cv_std
    }
])

display(comparison_results.round(3))


,Model,Features,Test MAE,Test RMSE,Test R2,CV MAE,CV Std
0,Original Random Forest,56,3.166,3.934,0.245,2.959,0.198
1,Random Forest Without Absences,55,3.347,4.202,0.139,3.178,0.296


### 7. Feature Selection Based on Importance

In [9]:

# Extract feature importance from original Random Forest
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": original_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Feature Importance Ranking")
print("--------------------------")
display(importance_df.head(20))


Feature Importance Ranking
--------------------------


,Feature,Importance
0,num__absences,0.269371
1,num__failures,0.241850
2,num__health,0.048028
3,num__goout,0.037610
4,num__age,0.030746
5,num__studytime,0.026094
6,num__Medu,0.021927
7,num__freetime,0.018140
8,num__Walc,0.017064
9,cat__Mjob_at_home,0.016795


### 8. Experiment C – Random Forest Using Top 20 Features

In [10]:

# Select top 20 features
top_20_features = importance_df.head(20)["Feature"].tolist()

# Get their column positions
top_20_indices = [
    list(feature_names).index(feature)
    for feature in top_20_features
]

# Create reduced datasets
X_train_top20 = X_train[:, top_20_indices]
X_test_top20 = X_test[:, top_20_indices]

print("Top 20 Feature Selection")
print("------------------------")
print("Original training shape:", X_train.shape)
print("Reduced training shape:", X_train_top20.shape)
print("Original testing shape:", X_test.shape)
print("Reduced testing shape:", X_test_top20.shape)
print("Selected features:", len(top_20_features))


Top 20 Feature Selection
------------------------
Original training shape: (316, 56)
Reduced training shape: (316, 20)
Original testing shape: (79, 56)
Reduced testing shape: (79, 20)
Selected features: 20


### 9. Train Random Forest – Top 20 Features

In [11]:

# Train Random Forest with top 20 features
rf_top20 = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42
)

rf_top20.fit(X_train_top20, y_train)

# Generate predictions
predictions_top20 = rf_top20.predict(X_test_top20)

# Evaluate model
top20_mae = mean_absolute_error(
    y_test, predictions_top20
)

top20_rmse = np.sqrt(
    mean_squared_error(y_test, predictions_top20)
)

top20_r2 = r2_score(
    y_test, predictions_top20
)

print("Random Forest - Top 20 Features")
print("--------------------------------")
print("MAE:", round(top20_mae, 3))
print("RMSE:", round(top20_rmse, 3))
print("R2 Score:", round(top20_r2, 3))


Random Forest - Top 20 Features
--------------------------------
MAE: 3.23
RMSE: 4.044
R2 Score: 0.202


### 10. Cross-Validation – Top 20 Features

In [12]:

cv_scores_top20 = cross_val_score(
    rf_top20,
    X_train_top20,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

top20_cv_mae = -cv_scores_top20.mean()
top20_cv_std = cv_scores_top20.std()

print("Top 20 Features - Cross-Validation")
print("----------------------------------")
print("Fold MAE Scores:", np.round(-cv_scores_top20, 3))
print("Mean CV MAE:", round(top20_cv_mae, 3))
print("CV Standard Deviation:", round(top20_cv_std, 3))


Top 20 Features - Cross-Validation
----------------------------------
Fold MAE Scores: [3.058 2.822 2.724 3.255 2.832]
Mean CV MAE: 2.938
CV Standard Deviation: 0.193


### 11. Complete Model Comparison

In [13]:

final_comparison = pd.DataFrame([
    {
        "Model": "Original Random Forest",
        "Features": 56,
        "Test MAE": baseline_results["MAE"],
        "Test RMSE": baseline_results["RMSE"],
        "Test R2": baseline_results["R2"],
        "CV MAE": 2.959,
        "CV Std": 0.198
    },
    {
        "Model": "RF Without Absences",
        "Features": 55,
        "Test MAE": no_abs_mae,
        "Test RMSE": no_abs_rmse,
        "Test R2": no_abs_r2,
        "CV MAE": no_abs_cv_mae,
        "CV Std": no_abs_cv_std
    },
    {
        "Model": "RF Top 20 Features",
        "Features": 20,
        "Test MAE": top20_mae,
        "Test RMSE": top20_rmse,
        "Test R2": top20_r2,
        "CV MAE": top20_cv_mae,
        "CV Std": top20_cv_std
    }
])

display(final_comparison.round(3))


,Model,Features,Test MAE,Test RMSE,Test R2,CV MAE,CV Std
0,Original Random Forest,56,3.166,3.934,0.245,2.959,0.198
1,RF Without Absences,55,3.347,4.202,0.139,3.178,0.296
2,RF Top 20 Features,20,3.230,4.044,0.202,2.938,0.193


### 12. Prediction Feature Availability Analysis

In [14]:

# Load original dataset for feature availability review
df_original = pd.read_csv(
    "../dataset/student-mat.csv",
    sep=";"
)

# Display all original feature names
feature_availability = pd.DataFrame({
    "Feature": df_original.columns,
    "Available_Before_Term": "Needs Review"
})

# Features explicitly excluded from model inputs
feature_availability.loc[
    feature_availability["Feature"].isin(["G1", "G2", "G3"]),
    "Available_Before_Term"
] = "Excluded / Target"

# Attendance is accumulated during the term
feature_availability.loc[
    feature_availability["Feature"] == "absences",
    "Available_Before_Term"
] = "Potentially Unavailable"

print("Prediction Feature Availability Review")
print("--------------------------------------")

display(feature_availability)


Prediction Feature Availability Review
--------------------------------------


,Feature,Available_Before_Term
0,school,Needs Review
1,sex,Needs Review
2,age,Needs Review
3,address,Needs Review
4,famsize,Needs Review
5,Pstatus,Needs Review
6,Medu,Needs Review
7,Fedu,Needs Review
8,Mjob,Needs Review
9,Fjob,Needs Review


In [15]:
# Classify features for beginning-of-term prediction

feature_categories = {
    "school": "Baseline profile",
    "sex": "Baseline profile",
    "age": "Baseline profile",
    "address": "Baseline profile",
    "famsize": "Baseline profile",
    "Pstatus": "Baseline profile",
    "Medu": "Baseline profile",
    "Fedu": "Baseline profile",
    "Mjob": "Baseline profile",
    "Fjob": "Baseline profile",
    "reason": "Baseline profile",
    "guardian": "Baseline profile",

    "failures": "Historical record",

    "traveltime": "Baseline questionnaire",
    "studytime": "Baseline questionnaire",
    "schoolsup": "Baseline questionnaire",
    "famsup": "Baseline questionnaire",
    "paid": "Baseline questionnaire",
    "activities": "Baseline questionnaire",
    "nursery": "Baseline questionnaire",
    "higher": "Baseline questionnaire",
    "internet": "Baseline questionnaire",
    "romantic": "Baseline questionnaire",
    "famrel": "Baseline questionnaire",
    "freetime": "Baseline questionnaire",
    "goout": "Baseline questionnaire",
    "Dalc": "Baseline questionnaire",
    "Walc": "Baseline questionnaire",
    "health": "Baseline questionnaire",

    "absences": "Unavailable at start - exclude",
    "G1": "Excluded - previous term grade",
    "G2": "Excluded - previous term grade",
    "G3": "Target variable"
}

feature_availability = pd.DataFrame({
    "Feature": df_original.columns,
    "Availability_Category": [
        feature_categories.get(col, "Needs review")
        for col in df_original.columns
    ]
})

display(feature_availability)

,Feature,Availability_Category
0,school,Baseline profile
1,sex,Baseline profile
2,age,Baseline profile
3,address,Baseline profile
4,famsize,Baseline profile
5,Pstatus,Baseline profile
6,Medu,Baseline profile
7,Fedu,Baseline profile
8,Mjob,Baseline profile
9,Fjob,Baseline profile


In [16]:
# Define beginning-of-term features

excluded_features = ["G1", "G2", "G3", "absences"]

early_features = [
    col for col in df_original.columns
    if col not in excluded_features
]

X_early = df_original[early_features]
y_early = df_original["G3"]

print("Beginning-of-Term Dataset")
print("-------------------------")
print("Number of features:", len(early_features))
print("Feature matrix shape:", X_early.shape)
print("Target shape:", y_early.shape)

print("\nExcluded features:", excluded_features)
print("\nRemaining features:")
print(early_features)

print("\nTarget distribution:")
display(y_early.describe())

Beginning-of-Term Dataset
-------------------------
Number of features: 29
Feature matrix shape: (395, 29)
Target shape: (395,)

Excluded features: ['G1', 'G2', 'G3', 'absences']

Remaining features:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']

Target distribution:


count    395.000000
mean      10.415190
std        4.581443
min        0.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       20.000000
Name: G3, dtype: float64

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify numerical and categorical features
numeric_features = X_early.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_early.select_dtypes(
    include=["object"]
).columns.tolist()

# Split the data
X_train_early, X_test_early, y_train_early, y_test_early = train_test_split(
    X_early,
    y_early,
    test_size=0.2,
    random_state=42
)

# Create preprocessing pipeline
early_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit only on training data
X_train_early_processed = early_preprocessor.fit_transform(X_train_early)

# Transform test data
X_test_early_processed = early_preprocessor.transform(X_test_early)

print("Beginning-of-Term Preprocessing")
print("--------------------------------")
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Training shape:", X_train_early_processed.shape)
print("Testing shape:", X_test_early_processed.shape)
print("Training target shape:", y_train_early.shape)
print("Testing target shape:", y_test_early.shape)

Beginning-of-Term Preprocessing
--------------------------------
Numerical features: 12
Categorical features: 17
Training shape: (316, 55)
Testing shape: (79, 55)
Training target shape: (316,)
Testing target shape: (79,)


C:\Users\vikra\AppData\Local\Temp\ipykernel_42164\1754027309.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_early.select_dtypes(


In [18]:
print("Original Random Forest Parameters")
print("----------------------------------")

for parameter, value in original_model.get_params().items():
    print(f"{parameter}: {value}")

Original Random Forest Parameters
----------------------------------
bootstrap: True
ccp_alpha: 0.0
criterion: squared_error
max_depth: 5
max_features: 1.0
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 3
min_samples_split: 5
min_weight_fraction_leaf: 0.0
monotonic_cst: None
n_estimators: 200
n_jobs: None
oob_score: False
random_state: 42
verbose: 0
warm_start: False


In [19]:
# Train beginning-of-term Random Forest
early_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=1.0,
    random_state=42
)

early_model.fit(
    X_train_early_processed,
    y_train_early
)

# Generate predictions
early_predictions = early_model.predict(
    X_test_early_processed
)

# Evaluate model
early_mae = mean_absolute_error(
    y_test_early,
    early_predictions
)

early_rmse = np.sqrt(
    mean_squared_error(y_test_early, early_predictions)
)

early_r2 = r2_score(
    y_test_early,
    early_predictions
)

print("Beginning-of-Term Random Forest Results")
print("----------------------------------------")
print(f"MAE:  {early_mae:.3f}")
print(f"RMSE: {early_rmse:.3f}")
print(f"R2:   {early_r2:.3f}")

Beginning-of-Term Random Forest Results
----------------------------------------
MAE:  3.342
RMSE: 4.197
R2:   0.141


In [20]:
# 5-fold cross-validation
early_cv_scores = cross_val_score(
    early_model,
    X_train_early_processed,
    y_train_early,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

early_cv_mae = -early_cv_scores

print("Beginning-of-Term Model: 5-Fold Cross-Validation")
print("------------------------------------------------")
print("Fold MAE scores:", np.round(early_cv_mae, 3))
print(f"Mean CV MAE: {early_cv_mae.mean():.3f}")
print(f"CV Standard Deviation: {early_cv_mae.std():.3f}")

Beginning-of-Term Model: 5-Fold Cross-Validation
------------------------------------------------
Fold MAE scores: [3.469 2.945 3.191 3.532 2.767]
Mean CV MAE: 3.181
CV Standard Deviation: 0.295


In [21]:
# Save beginning-of-term model and preprocessing artifacts

joblib.dump(
    early_model,
    "../artifacts/beginning_term_model.pkl"
)

joblib.dump(
    early_preprocessor,
    "../artifacts/beginning_term_preprocessor.pkl"
)

early_model_metrics = {
    "Model": "Beginning-of-Term Random Forest",
    "Features": len(early_features),
    "Transformed Features": X_train_early_processed.shape[1],
    "MAE": early_mae,
    "RMSE": early_rmse,
    "R2": early_r2,
    "CV_MAE": early_cv_mae.mean(),
    "CV_Std": early_cv_mae.std()
}

joblib.dump(
    early_model_metrics,
    "../artifacts/beginning_term_model_metrics.pkl"
)

print("Beginning-of-term model saved successfully.")
print("Saved files:")
print("- beginning_term_model.pkl")
print("- beginning_term_preprocessor.pkl")
print("- beginning_term_model_metrics.pkl")

Beginning-of-term model saved successfully.
Saved files:
- beginning_term_model.pkl
- beginning_term_preprocessor.pkl
- beginning_term_model_metrics.pkl


In [22]:
# Reload saved artifacts
loaded_early_model = joblib.load(
    "../artifacts/beginning_term_model.pkl"
)

loaded_early_preprocessor = joblib.load(
    "../artifacts/beginning_term_preprocessor.pkl"
)

loaded_early_metrics = joblib.load(
    "../artifacts/beginning_term_model_metrics.pkl"
)

# Transform test data using the reloaded preprocessor
X_test_reloaded = loaded_early_preprocessor.transform(
    X_test_early
)

# Generate predictions using the reloaded model
reloaded_predictions = loaded_early_model.predict(
    X_test_reloaded
)

# Validate predictions
print("Saved Model Validation")
print("----------------------")
print("Model:", type(loaded_early_model).__name__)
print("Predictions generated:", len(reloaded_predictions))

print(
    "Predictions match:",
    np.allclose(early_predictions, reloaded_predictions)
)

print("\nReloaded Model Metrics:")
print(loaded_early_metrics)

Saved Model Validation
----------------------
Model: RandomForestRegressor
Predictions generated: 79
Predictions match: True

Reloaded Model Metrics:
{'Model': 'Beginning-of-Term Random Forest', 'Features': 29, 'Transformed Features': 55, 'MAE': 3.34231531511462, 'RMSE': np.float64(4.197195390099962), 'R2': 0.14087254165408658, 'CV_MAE': np.float64(3.1808724118881453), 'CV_Std': np.float64(0.29466857156371523)}
